In [0]:
%run ./data_utils

#ukey merge

In [0]:
def cleanse_name_string(name):
    if name is None or str(name).strip() == "":
        return "" 
    else:
        return name.lower()


def email_match_key(email: str) -> str:
    if email is None or email.strip() == "":
        return None
    return email.lower()

def phone_match_key(phone: str) -> str:
    if phone is None or phone.strip() == "":
        return None
    return phone.lower()

def address_match_key(addr1: str, addr2: str, addr3: str, city: str, postal: str) -> str:
    parts = []
    for part in [addr1, addr2, addr3, city, postal]:
        if part is not None and part.strip() != "":
            parts.append(part.strip().lower())
    
    address_key = "".join(parts)
    if address_key == "":
        return None
    return address_key



cleanse_name_udf = F.udf(cleanse_name_string, StringType())
email_match_udf = F.udf(email_match_key, StringType())
phone_match_udf = F.udf(phone_match_key, StringType())
address_match_udf = F.udf(address_match_key, StringType())

# cleanse_full_name_udf = F.udf(cleanse_full_name_string, StringType())
# full_name_match_udf = F.udf(full_name_match_key, StringType())

In [0]:
# ─────────────────────────────────────────────
# 辅助函数
# ─────────────────────────────────────────────
def is_blank(value) -> bool:
    """判断字符串是否为空（None、空字符串、字面量 'null' 均视为空）"""
    if value is None:
        return True
    return str(value).strip() == "" or str(value).lower() == "null"


def cleanse_full_name_string(firstname, lastname, fullname) -> tuple[str, str]:
    """
    清洗姓名字段，返回 (升序匹配键, 降序匹配键) 两个值。

    处理规则：
      1. None / 空 / "null" 统一转为空字符串, 移除全角空格 U+3000(日文场景)
      2. firstname / lastname 移除所有空白字符
      3. fullname 将连续空白压缩为单个空格, 去掉首位空格

      4. 按优先级取值: firstname+lastname → fullname → firstname或lastname → None
        4.1  fullname 超过 2 个词时，移除全部空格
        4.2  fullname 不超过 2 个词时，使用空格切分
        4.3  去重name

      5. 分别升序/降序排序拼接
    """
    # Step 1: 空值归一化
    firstname = "" if is_blank(firstname) else firstname
    lastname  = "" if is_blank(lastname)  else lastname
    fullname  = "" if is_blank(fullname)  else fullname

    # Step 2 & 3: 清理空白, 移除全角空格 U+3000 (日文场景)
    firstname = re.sub(r'\s+', '',  firstname).lower().replace('\u3000', '')          # 移除所有空白
    lastname  = re.sub(r'\s+', '',  lastname).lower().replace('\u3000', '')           # 移除所有空白
    fullname  = re.sub(r'\s+', ' ', fullname).lower().replace('\u3000', '').strip()   # 压缩为单空格

    # Step 4: 按优先级取值
    unique_words = set()

    if firstname and lastname:
        # Branch 1: 姓和名都有值
        unique_words.add(firstname)
        unique_words.add(lastname)

    elif fullname:
        # Branch 2: 只有 fullname 有值
        # fullname 超过 2 词则不进行姓名排序
        if len(fullname.split(" ")) > 2:
            fullname = re.sub(r'\s+', '', fullname)
            unique_words.add(fullname)
        else:
            unique_words.update(fullname.split(" "))
            
    elif firstname or lastname:
        # Branch 3: 姓或名只有其中一个有值
        if firstname:
            unique_words.add(firstname)
        else:    
            unique_words.add(lastname)
    else:
        # Branch 4: 全部为空
        return (None, None)

    # Step 5: 分别升序/降序排序拼接
    sorted_asc   = "".join(sorted(unique_words))
    sorted_desc  = "".join(sorted(unique_words, reverse=True))
 
    return (sorted_asc, sorted_desc)


def full_name_sort_match_key(
        eng_firstname, eng_lastname, eng_fullname,
        lcl_firstname, lcl_lastname, lcl_fullname,
        lcl_firstname2=None, lcl_lastname2=None, lcl_fullname2=None) -> tuple[str, str]:
    """
    根据优先级选取有效姓名，直接返回 (升序匹配键, 降序匹配键)

    优先级: 本地名1 → 本地名2 → 英文名 → None
    优先级判断仅依据升序（正常）名称是否为空。

    lcl_*2 传 None 时跳过本地名2,等同于 V2(6参数)行为。
    """
    # 分别清洗三组姓名，各自得到 (升序键, 降序键)
    asc_lcl,  desc_lcl  = cleanse_full_name_string(lcl_firstname,  lcl_lastname,  lcl_fullname)
    asc_lcl2, desc_lcl2 = cleanse_full_name_string(lcl_firstname2, lcl_lastname2, lcl_fullname2)
    asc_eng,  desc_eng  = cleanse_full_name_string(eng_firstname,  eng_lastname,  eng_fullname)

    # 仅根据升序名称（正常名称）判断优先级，同时返回升序和降序键
    if not is_blank(asc_lcl):
        return (asc_lcl,  desc_lcl)

    elif not is_blank(asc_lcl2):
        return (asc_lcl2, desc_lcl2)

    elif not is_blank(asc_eng):
        return (asc_eng,  desc_eng)

    else:
        return (None, None)


def _udf_v2(ef, el, efu, lf, ll, lfu):
    asc, desc = full_name_sort_match_key(ef, el, efu, lf, ll, lfu)
    return (asc, desc)

def _udf_v3(ef, el, efu, lf, ll, lfu, lf2, ll2, lfu2):
    asc, desc = full_name_sort_match_key(ef, el, efu, lf, ll, lfu, lf2, ll2, lfu2)
    return (asc, desc)


SORT_MATCH_KEY_SCHEMA = StructType([
    StructField("asc",  StringType(), nullable=True),
    StructField("desc", StringType(), nullable=True),
])

full_name_sort_match_key_v2_udf = F.udf(_udf_v2, SORT_MATCH_KEY_SCHEMA)  # V2
full_name_sort_match_key_v3_udf = F.udf(_udf_v3, SORT_MATCH_KEY_SCHEMA)  # V3

#Cid Mapping

In [0]:
# def cid_email_match_key(email: str) -> str:
#     if email is None or email.strip() == "":
#         return ""
#     return email.lower()

# def cid_phone_match_key(phone: str) -> str:
#     if phone is None or phone.strip() == "":
#         return ""
#     return phone.lower()

def cid_email_match_key(email: str) -> str:
    if email is None :
        return None
    
    if email.strip() == "":
        return ""
    
    return email.lower()

def cid_phone_match_key(phone: str) -> str:
    if phone is None :
        return None
    
    if phone.strip() == "":
        return ""
    
    return phone.lower()


cid_email_match_udf = F.udf(cid_email_match_key, StringType())
cid_phone_match_udf = F.udf(cid_phone_match_key, StringType())

In [0]:
def build_cid_mapping_json_str(
    tran_action,
    tran_srcs_code,
    mapping_srcs_code,
    tran_mrkt_code,
    tran_brnd_code,
    tran_order_id,
    tran_mapping_conusmer_id,
    document_timestamp,
    document_uuid,
    record_uuid,
    mapping_timestamp,
):
    """
    构建 ConsumerMappingList JSON 字符串
    """

    data = {
        "ConsumerMappingList": {
            "Header": {
                "@Action": tran_action,
                "DocumentTimestamp": document_timestamp,
                "DocumentUUID": document_uuid,
            },
            "ConsumerMapping": {
                "@RecordUUID": record_uuid,
                "MappingTimestamp": mapping_timestamp,
                "MasterConsumer": {
                    "@TypeCode": tran_srcs_code,
                    "MarketCode": tran_mrkt_code,
                    "BrandCode": tran_brnd_code,
                    "Id": tran_order_id,
                },
                "MappingConsumerList": {
                    "MappingConsumer": {
                        "@Code": mapping_srcs_code,
                        "MarketCode": tran_mrkt_code,
                        "BrandCode": tran_brnd_code,
                        "Id": tran_mapping_conusmer_id,
                    }
                },
            },
        }
    }
    return json.dumps(data, ensure_ascii=False)

build_cid_mapping_json_udf = F.udf(build_cid_mapping_json_str, StringType())